In [ ]:
import os

# Send diagnostic plots to a temporary folder; the paper renderer handles final assets.
from pathlib import Path as _NotebookPath
import tempfile as _notebook_tempfile
NOTEBOOK_FIGURE_DIR = _NotebookPath(
    os.environ.get("ANNALS_NOTEBOOK_FIGURE_DIR")
    or _notebook_tempfile.mkdtemp(prefix="annals-frechet_family-")
)
NOTEBOOK_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
import numpy as np
import matplotlib.pyplot as plt

from scipy.special import gamma
from scipy.stats import norm, invweibull

# Avoid Matplotlib cache warnings on some setups
os.environ.setdefault("MPLCONFIGDIR", os.path.join(os.getcwd(), ".matplotlib"))

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 130,
    "axes.grid": True,
})

rng = np.random.default_rng(0)



In [ ]:
def gaussian_to_frechet_1d(x: np.ndarray, alpha: float, eps: float = 1e-12) -> np.ndarray:
    """Monotone 1D pushforward: x~N(0,1) -> y~Fréchet(alpha) via quantiles.

    Fréchet(alpha) with CDF exp(-y^{-alpha}) on y>0.
    In SciPy: invweibull(c=alpha), loc=0, scale=1.
    """
    u = norm.cdf(x)
    u = np.clip(u, eps, 1.0 - eps)
    return invweibull.ppf(u, c=alpha)


def jacobian_diag_entry_1d(x: np.ndarray, alpha: float, eps: float = 1e-12) -> np.ndarray:
    """Derivative d/dx of y = frechet_ppf(Phi(x))."""
    y = gaussian_to_frechet_1d(x, alpha=alpha, eps=eps)
    return norm.pdf(x) / invweibull.pdf(y, c=alpha)


def transport_2d(X: np.ndarray, alpha: float, eps: float = 1e-12) -> np.ndarray:
    """2D diagonal transport: applies 1D map to each coordinate.

    X: (...,2)
    returns Y: (...,2)
    """
    X = np.asarray(X)
    assert X.shape[-1] == 2
    Y = np.empty_like(X, dtype=float)
    Y[..., 0] = gaussian_to_frechet_1d(X[..., 0], alpha=alpha, eps=eps)
    Y[..., 1] = gaussian_to_frechet_1d(X[..., 1], alpha=alpha, eps=eps)
    return Y


def jacobian_2d(X: np.ndarray, alpha: float, eps: float = 1e-12) -> np.ndarray:
    """Jacobian matrices J(x) for the 2D diagonal transport.

    returns J with shape (..., 2, 2)
    """
    X = np.asarray(X)
    assert X.shape[-1] == 2
    j1 = jacobian_diag_entry_1d(X[..., 0], alpha=alpha, eps=eps)
    j2 = jacobian_diag_entry_1d(X[..., 1], alpha=alpha, eps=eps)

    J = np.zeros(X.shape[:-1] + (2, 2), dtype=float)
    J[..., 0, 0] = j1
    J[..., 1, 1] = j2
    return J


def log_abs_det_jacobian_2d(X: np.ndarray, alpha: float, eps: float = 1e-12) -> np.ndarray:
    X = np.asarray(X)
    j1 = jacobian_diag_entry_1d(X[..., 0], alpha=alpha, eps=eps)
    j2 = jacobian_diag_entry_1d(X[..., 1], alpha=alpha, eps=eps)
    return np.log(np.abs(j1)) + np.log(np.abs(j2))



In [ ]:
# --- Plots: 1D Jacobian entry j_alpha(x) ---

alphas = [2.1, 3.0, 5.0, 10.0, 30.0]

x = np.linspace(-4.0, 4.0, 2001)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), constrained_layout=True)

ax = axes[0]
for a in alphas:
    j = jacobian_diag_entry_1d(x, alpha=a)
    ax.plot(x, j, label=f"alpha={a}")
ax.set_title(r"Jacobian diagonal entry $j_\alpha(x)$")
ax.set_xlabel("x")
ax.set_ylabel(r"$j_\alpha(x)$")
ax.set_yscale("log")
ax.legend(ncol=2, fontsize=8)

ax = axes[1]
for a in alphas:
    j = jacobian_diag_entry_1d(x, alpha=a)
    ax.plot(x, np.log(j), label=f"alpha={a}")
ax.set_title(r"$\log j_\alpha(x)$")
ax.set_xlabel("x")
ax.set_ylabel(r"$\log j_\alpha(x)$")
ax.legend(ncol=2, fontsize=8)

plt.show()



In [ ]:
# --- Plots: 2D heatmaps of log|det J_T(x)| over a grid ---

grid_lim = 4.0
n_grid = 301
x1 = np.linspace(-grid_lim, grid_lim, n_grid)
x2 = np.linspace(-grid_lim, grid_lim, n_grid)
X1, X2 = np.meshgrid(x1, x2, indexing="xy")
Xgrid = np.stack([X1, X2], axis=-1)  # (n_grid, n_grid, 2)

fig, axes = plt.subplots(1, len(alphas), figsize=(3.2 * len(alphas), 3.1), constrained_layout=True)
if len(alphas) == 1:
    axes = [axes]

# precompute global color limits for comparability
vals = []
for a in alphas:
    vals.append(log_abs_det_jacobian_2d(Xgrid, alpha=a))
allv = np.concatenate([v.ravel() for v in vals])
vmin, vmax = np.quantile(allv, [0.02, 0.98])

for ax, a, Z in zip(axes, alphas, vals):
    im = ax.imshow(
        Z,
        origin="lower",
        extent=[-grid_lim, grid_lim, -grid_lim, grid_lim],
        cmap="coolwarm",
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest",
        aspect="equal",
    )
    ax.set_title(rf"$\zeta={a}$", fontsize=16)

cbar = fig.colorbar(im, ax=axes, shrink=0.9)
cbar.set_label(r"$\log|\det J_{T}|$", fontsize=16)


plt.show()
fig.savefig(NOTEBOOK_FIGURE_DIR / "logdet_heatmaps_frechet.png", dpi=150, bbox_inches="tight")



In [ ]:
from numpy.polynomial.hermite import hermgauss


def log_jacobian_diag_entry_1d(x: np.ndarray, alpha: float, eps: float = 1e-12) -> np.ndarray:
    """Stable log of j_alpha(x) = phi(x) / frechet_pdf(frechet_ppf(Phi(x)))."""
    u = norm.cdf(x)
    u = np.clip(u, eps, 1.0 - eps)
    y = invweibull.ppf(u, c=alpha)
    return norm.logpdf(x) - invweibull.logpdf(y, c=alpha)


def normal_expectation_gauss_hermite_1d(fun, n: int = 80) -> float:
    """Approximate E[fun(Z)] for Z~N(0,1) using n-point Gauss–Hermite."""
    xs, ws = hermgauss(n)  # integrates against exp(-x^2)
    z = np.sqrt(2.0) * xs
    vals = fun(z)
    return float(np.sum(ws * vals) / np.sqrt(np.pi))


def estimate_E_sum_sq_jac_diag_2d(alpha: float, n_gh: int = 80) -> float:
    """Estimate E_{X~N(0,I2)}[ j1(X)^2 + j2(X)^2 ] for diagonal map.

    Since coords are iid and Jacobian is diag(j(x1), j(x2)):
      E[j1^2 + j2^2] = 2 * E_{Z~N(0,1)}[j(Z)^2].
    """
    def j2_of_z(z):
        logj = log_jacobian_diag_entry_1d(z, alpha=alpha)
        return np.exp(np.clip(2.0 * logj, -745.0, 709.0))

    Ej2 = normal_expectation_gauss_hermite_1d(j2_of_z, n=n_gh)
    return 2.0 * Ej2



In [ ]:
# --- Plot: estimated E_gaussian[ sum of squares of Jacobian diagonals ] vs alpha ---

# Fréchet shape grid (denser near heavy-tailed regime alpha->2+)
alpha_grid = np.concatenate([
    np.linspace(2.05, 5.0, 30),
    np.linspace(5.0, 20.0, 31)[1:],
    np.linspace(20.0, 80.0, 25)[1:],
])

n_gh = 140
rhs_est = np.array([estimate_E_sum_sq_jac_diag_2d(a, n_gh=n_gh) for a in alpha_grid])

# "LHS" overlay: variance of the 2D iid Fréchet(alpha) target (trace of covariance)
# For Fréchet(alpha) with CDF exp(-y^{-alpha}):
#   E[Y^p] = Gamma(1 - p/alpha) for p < alpha.
# Hence Var(Y) = Gamma(1-2/alpha) - Gamma(1-1/alpha)^2, for alpha>2.
mu1 = gamma(1.0 - 1.0 / alpha_grid)
mu2 = gamma(1.0 - 2.0 / alpha_grid)
var1 = mu2 - mu1**2
lhs_var_target = 2.0 * var1

fig, ax = plt.subplots(1, 1, figsize=(7.9, 6.9), constrained_layout=True)
ax.plot(
    alpha_grid,
    rhs_est,
    marker="o",
    markersize=2.5,
    linewidth=1.2,
    label=rf"Jacobian Size",
)
ax.plot(
    alpha_grid,
    lhs_var_target,
    linewidth=2.0,
    linestyle="-",
    color="#d62728",
    label=r"Lower Bound",
)

# ax.set_title(r"Poincaré comparison", fontsize=18)
ax.set_xlabel(r"shape parameter $\zeta$", fontsize=22)
ax.set_ylabel(r"size", fontsize=22)
ax.set_yscale("log")
ax.tick_params(axis="both", which="major", labelsize=14)
ax.legend(fontsize=22)

plt.show()
fig.savefig(NOTEBOOK_FIGURE_DIR / "poincare_comparison_frechet.png", dpi=180)

# # Optional quick Monte Carlo check on a few alphas
# check = [2.1, 2.5, 3.0, 5.0, 10.0, 40.0]
# Nmc = 600_000
# Z = rng.standard_normal(Nmc)
# for a in check:
#     logj = log_jacobian_diag_entry_1d(Z, alpha=a)
#     j2 = np.exp(np.clip(2.0 * logj, -745.0, 709.0))
#     mc = 2.0 * float(np.mean(j2))
#     gh = estimate_E_sum_sq_jac_diag_2d(a, n_gh=n_gh)
#     print(f"alpha={a:5.1f}  GH(n={n_gh})={gh: .6e}   MC(N={Nmc})={mc: .6e}   ratio(MC/GH)={mc/gh: .4f}")

